# 🏋️‍♂️ Multi-Agent RAG with Microsoft Agent Framework and Azure AI Foundry 🥑

Welcome to this fun, self-guided workshop where we build a multi-agent Retrieval-Augmented Generation (RAG) pipeline using the Microsoft Agent Framework style with the new Azure AI Foundry portal workflow. Our team of agents collaborates to answer fitness and health questions in an engaging way.

## 1. Setup

You'll import the required libraries and create a Foundry project client plus an OpenAI-compatible client using the newer endpoint-based flow. Authentication uses Microsoft Entra ID (keyless) via `DefaultAzureCredential`, so ensure `PROJECT_ENDPOINT` and `MODEL_DEPLOYMENT_NAME` are set in your `.env` and you are signed in (`az login`).

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


# Load environment variables
notebook_path = Path().absolute()
env_path = notebook_path.parent.parent / '.env'  # Adjust path as needed
load_dotenv(env_path)

project_endpoint = os.environ["PROJECT_ENDPOINT"]
model_name = os.environ["MODEL_DEPLOYMENT_NAME"]

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

print("✅ Foundry project and OpenAI clients created successfully!")

✅ Foundry project and OpenAI clients created successfully!


## 2. Create Sample Health Data and Retrieval Tool

We'll define a small list of health tips and a simple retrieval function. This function simulates retrieving relevant health tips based on keywords in the user's query.

In [2]:
# Define sample health tips
health_tips = [
    {"id": "tip1", "content": "Do a 10-minute HIIT workout to boost your metabolism.", "source": "Fitness Guru"},
    {"id": "tip2", "content": "Take a brisk 15-minute walk to clear your mind and improve circulation.", "source": "Health Coach"},
    {"id": "tip3", "content": "Stretch for 5 minutes every hour if you're sitting at a desk.", "source": "Wellness Expert"},
    {"id": "tip4", "content": "Incorporate strength training twice a week for overall fitness.", "source": "Personal Trainer"},
    {"id": "tip5", "content": "Drink water regularly to stay hydrated during workouts.", "source": "Nutritionist"}
]

def retrieve_tips(query: str) -> str:
    """Return health tips whose content contains keywords from the query."""
    query_lower = query.lower()
    relevant = []
    for tip in health_tips:
        # Check if any word in the query is in the tip content
        if any(word in tip["content"].lower() for word in query_lower.split()):
            relevant.append(f"Source: {tip['source']} => {tip['content']}")
    if not relevant:
        # If no tips match, return all tips (for demo purposes)
        relevant = [f"Source: {tip['source']} => {tip['content']}" for tip in health_tips]
    return "\n".join(relevant)

print("✅ Sample health tips and retrieval tool created!")

✅ Sample health tips and retrieval tool created!


## 3. Define Our Multi-Agent RAG Pipeline

We will create two Foundry agents:

1. RetrieverAgent: organizes the retrieved fitness and health tips for the query.
2. ResponderAgent: crafts a fun, engaging answer using the retriever output and user question.

These two agents are orchestrated sequentially to simulate collaborative problem-solving.

In [3]:
from azure.ai.projects.models import PromptAgentDefinition


def create_agent(agent_name: str, instructions: str):
    return project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=instructions,
        ),
        description=f"{agent_name} for multi-agent RAG fitness demo.",
    )


retriever_agent = create_agent(
    "maf-retriever-agent",
    (
        "You are RetrieverAgent. You receive a user question and retrieved tip context. "
        "Return a concise, useful summary of only the relevant tips."
    ),
)

responder_agent = create_agent(
    "maf-responder-agent",
    (
        "You are ResponderAgent, a friendly fitness coach. Use the retriever summary to craft "
        "an engaging answer. Include practical suggestions and a brief health disclaimer."
    ),
)

print("✅ Multi-agent RAG pipeline defined!")

✅ Multi-agent RAG pipeline defined!


## 4. Try a Query

Let's test our multi-agent RAG system with a fun fitness query. For example:

> **User Query:** _I'm very busy but want to stay fit. What quick exercises can I do?_

In [4]:
from openai import NotFoundError

RETRIEVER_NAME = "maf-retriever-agent"
RESPONDER_NAME = "maf-responder-agent"

RETRIEVER_INSTRUCTIONS = (
    "You are RetrieverAgent. You receive a user question and retrieved tip context. "
    "Return a concise, useful summary of only the relevant tips."
)
RESPONDER_INSTRUCTIONS = (
    "You are ResponderAgent, a friendly fitness coach. Use the retriever summary to craft "
    "an engaging answer. Include practical suggestions and a brief health disclaimer."
)


def recreate_agents() -> None:
    global retriever_agent, responder_agent
    retriever_agent = create_agent(RETRIEVER_NAME, RETRIEVER_INSTRUCTIONS)
    responder_agent = create_agent(RESPONDER_NAME, RESPONDER_INSTRUCTIONS)


def run_with_agent(agent_name: str, prompt: str) -> str:
    return openai_client.responses.create(
        input=prompt,
        extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
    ).output_text


user_query = "I'm very busy but want to stay fit. What quick exercises can I do?"
retrieved_context = retrieve_tips(user_query)
print(f"User: {user_query}")

retriever_input = (
    f"User query:\n{user_query}\n\n"
    f"Retrieved tips:\n{retrieved_context}\n\n"
    "Summarize the most relevant tips for the user."
)

try:
    retriever_text = run_with_agent(retriever_agent.name, retriever_input)
except NotFoundError:
    recreate_agents()
    retriever_text = run_with_agent(retriever_agent.name, retriever_input)

print(f"RetrieverAgent: {retriever_text}")

responder_input = (
    f"User query:\n{user_query}\n\n"
    f"Retriever output:\n{retriever_text}\n\n"
    "Create a friendly final answer for the user."
)
responder_text = run_with_agent(responder_agent.name, responder_input)
print(f"ResponderAgent: {responder_text}")

for agent in (retriever_agent, responder_agent):
    project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ Cleaned up agent versions.")

User: I'm very busy but want to stay fit. What quick exercises can I do?
RetrieverAgent: If you're busy, focus on fast, effective options:

- Do a 10-minute HIIT workout for a quick fitness and metabolism boost.
- Take a brisk 15-minute walk when you can for easy cardio and energy.
- If you sit a lot, stretch for 5 minutes every hour to stay loose and active.
- Add strength training twice a week to maintain overall fitness.
- Stay hydrated by drinking water regularly, especially around exercise.
ResponderAgent: Absolutely — if you’re busy, the key is to make fitness **short, simple, and consistent**.

Here are some quick options that work well:

- **10-minute HIIT workout:** Great for a fast cardio and metabolism boost. Think squats, jumping jacks, mountain climbers, push-ups, or high knees in short intervals.
- **Brisk 15-minute walk:** Easy to fit in during a break, after meals, or between tasks. It helps with energy, heart health, and stress.
- **5 minutes of stretching every hour:*

## 5. Conclusion

In this notebook, we built a fun multi-agent Retrieval-Augmented Generation pipeline using the newer Foundry v2 agent workflow with a fitness and health theme. We created a RetrieverAgent that organizes relevant health tips and a ResponderAgent that crafts an engaging answer to the user's query.

Feel free to modify and expand this notebook to explore more advanced multi-agent collaborations. Happy coding and stay fit! 💪🥦